# Lab 2: Data Discovery and Exploration with Snowflake CoCo

You have just been hired at a sports analytics firm that specializes in American football. The firm gathers data and statistics about games and players in order to make predictions and advise teams.

Since you are new to the firm, you are not yet familiar with their data. What data do they track? How is it organized? What tables do they use?

Fortunately, your new employer uses Snowflake and has given you access to Snowflake CoCo.

**Objective**: In this lab exercise, you will use CoCo's natural-language SQL generation and catalog awareness to explore the Professional Football League (PFL) dataset without memorizing table or column names.


## Setup

First, let's get your environment set up. Run the cell below to generate the PFL dataset - a database of information about the fictional Professional Football League.

In [ ]:
from resources.setup_pfl import setup_pfl
setup_pfl()

The database `PFL_DB` has been created with the schema `STATS_AND_INFO` that holds all of the tables and views relating to the Professional Football League.

Run the cell below to set your context to this database and schema.

In [ ]:
%%sql -r Set_Context
CREATE WAREHOUSE IF NOT EXISTS COCOLABS_WH WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE COCOLABS_WH;
USE DATABASE PFL_DB;
USE SCHEMA STATS_AND_INFO;

## Exploring the Data Environment

When getting started with an unfamiliar database, it may be tempting to browse through its tables and views using Database Explorer. This is not a bad idea, but it can get confusing. Table names may not give the full picture of what is in the table and figuring out how tables join together is a task in itself.

Let's use Snowflake CoCo instead.

### Open Snowflake CoCo

If it is not already open, click the Snowflake CoCo button on the bottom right of the screen to open the CoCo panel.

<div style="max-width:300px;">

![](resources/img/cortex-code-button.png)</div>

### Explore the tables

Enter the following prompt into Snowflake CoCo and press enter:

<code style="display:block; user-select:all; padding: 15px;">
Provide a brief description of all the tables in PFL_DB.STATS_AND_INFO along with their row counts.
</code>

How many tables are there?

Which table shows information about each team?

### Explore the views

The database also contains a number of views. 

Enter the following prompt into Snowflake CoCo and press enter:

<code style="display:block; user-select:all; padding: 15px;">
Provide a brief description of all the views in PFL_DB.STATS_AND_INFO.
</code>

This shows a nice summary of each view. But you are curious about which tables the views are pulling their data from. Instead of parsing through some complex SQL, ask Snowflake CoCo. Enter the following prompt and press Enter:

<code style="display:block; user-select:all; padding: 15px;">
What tables does V_PASSING_LEADERS pull data from?
</code>

### Finding a table by description

You are interested in learning more about which colleges team members played for. None of the table names include the words "college" or "university", and it would be difficult to look at the columns of each table. 

Instead, ask Snowflake CoCo with the following prompt:

<code style="display:block; user-select:all; padding: 15px;">
Which tables will show what college a player previously played for?
</code>

### Understanding the relationship between tables

Tables can often be joined together. The relationships between them can be complex and are not always obvious. Snowflake CoCo can quickly show the relationships. Use the following prompt:


<code style="display:block; user-select:all; padding: 15px;">
Which tables can be joined together and which ones are the key "hub" tables?
</code>

## Querying Data for Insights

You realize that you have a meeting coming up and are expected to present some statistical insights on the data you've explored so far. But you've barely had a chance to get familiar with the data layout and your SQL skills are not too strong. How will you be able to impress your new coworkers?

You've decided to determine how much the top performing players are paid and whether they are paid proportionally to their skill.

### Find the best players

Finding this information may take a few steps. Plus, it can be fun to conversationally narrow in on the data you are looking for. For now, let's just find the five best players.

Use the following prompt:

<code style="display:block; user-select:all; padding: 15px;">
Who are the 5 best players?
</code>

Examine the output, including the parts where Snowflake CoCo shows its reasoning. How did CoCo determine the answer to this question? Did it find the players with the most touchdowns? With the most total yards gained? Most tackles? There are a lot of different ways to be the "best" player.

Although Snowflake CoCo can do quite a bit, you still need to guide it into making decisions. The prompt above is vague. Unless it asked you for clarification, CoCo likely made some sort of choice of how to interpret it. But was it the correct choice?

### Find a good statistic

Let's see what types of statistics are available so that we can determine a good one to use as a metric.

Use the following prompt:


<code style="display:block; user-select:all; padding: 15px;">
What types of statistics are stored and what tables can I find them in?
</code>

Examine the output. The `PLAYER_SEASON_STATS` table has a lot of good statistics on individual players. Instead of asking for a vague "best" player, let's focus in on which player passed the most yards.


### Using an `@` mention

There are a few tables that contained statistics. Although Snowflake CoCo is usually good at determining the correct tables to draw information from, it can be helpful to be more specific, especially if you have tables with similar data.

Use `@OBJECT_NAME` to specify an object in Snowflake, like a table.

Use the following prompt to find the 5 best players based on who passed the most yards.

<code style="display:block; user-select:all; padding: 15px;">
Using @PLAYER_SEASON_STATS, which 5 players have the highest pass yards?
</code>

### Get some more analytical data

Now that we have gotten our query on the right track, let's expand it. It's time to ask Snowflake CoCo to do something with a bit more analysis. Let's compare the amount of yards passed with the player's salary and see how much players get paid per yard.

Use the following prompt:

<code style="display:block; user-select:all; padding: 15px;">
Using @PLAYER_SEASON_STATS, which 5 players have the highest pass yards? 
Also, show how much they are paid and calculate how much they are paid per yard.
</code>

Which player gets paid the most per yard? 

Which player is the best "value" by being paid the least per yard?

### Save the SQL query

You've decided that this information would be great to present at your upcoming meeting. In fact, it might be helpful to run this query on a regular basis as the data gets updated.

In the Snowflake CoCo panel, you should see a box shortly after your prompt that has a few buttons:

<div style="max-width:600px;">

![](resources/img/cortex-sql-box.png)</div>

This box may be labeled something different than what is shown here, but it is likely similar.

Click the expand button ( ![](resources/img/expand-sql-box.png) ) on the box.

This box shows the SQL code that CoCo generated in order to gather the information. You can now run this code whenever you like instead of asking CoCo to recreate it every time.

Copy and paste the SQL code from the CoCo panel into the SQL cell below, then run the SQL cell.

In [ ]:
%%sql -r Cost_per_Yard_SQL
SELECT
    p.FIRST_NAME || ' ' || p.LAST_NAME AS PLAYER_NAME,
    t.FULL_NAME AS TEAM,
    SUM(pss.PASS_YARDS) AS TOTAL_PASS_YARDS,
    SUM(pss.PASS_TOUCHDOWNS) AS TOTAL_PASS_TDS,
    pc.TOTAL_VALUE,
    pc.ANNUAL_SALARY,
    ROUND(pc.TOTAL_VALUE / NULLIF(SUM(pss.PASS_YARDS), 0), 2) AS COST_PER_YARD
FROM PFL_DB.STATS_AND_INFO.PLAYER_SEASON_STATS pss
JOIN PFL_DB.STATS_AND_INFO.PLAYERS p ON pss.PLAYER_ID = p.PLAYER_ID
JOIN PFL_DB.STATS_AND_INFO.TEAMS t ON pss.TEAM_ID = t.TEAM_ID
JOIN PFL_DB.STATS_AND_INFO.PLAYER_CONTRACTS pc ON pss.PLAYER_ID = pc.PLAYER_ID AND pc.IS_ACTIVE = TRUE
GROUP BY p.FIRST_NAME, p.LAST_NAME, t.FULL_NAME, pc.TOTAL_VALUE, pc.ANNUAL_SALARY
ORDER BY TOTAL_PASS_YARDS DESC
LIMIT 5;

### Use CoCo instead

Did we really just manually copy and paste? This is the era of agentic AI! Let's make CoCo do the work instead.

Use the following prompt to tell CoCo to copy the information into the cell below:

<code style="display:block; user-select:all; padding: 15px;">
Copy the SQL that you used to generate that information into the Cost_per_Yard_SQL_copied_by_CoCo notebook cell.
</code>

When CoCo edits the cell, it will ask you to confirm the changes. Click **Keep all**.

In [ ]:
%%sql -r Cost_per_Yard_SQL_copied_by_Cortex
SELECT
    p.FIRST_NAME || ' ' || p.LAST_NAME AS PLAYER_NAME,
    t.FULL_NAME AS TEAM,
    SUM(pss.PASS_YARDS) AS TOTAL_PASS_YARDS,
    SUM(pss.PASS_TOUCHDOWNS) AS TOTAL_PASS_TDS,
    pc.TOTAL_VALUE,
    pc.ANNUAL_SALARY,
    ROUND(pc.TOTAL_VALUE / NULLIF(SUM(pss.PASS_YARDS), 0), 2) AS COST_PER_YARD
FROM PFL_DB.STATS_AND_INFO.PLAYER_SEASON_STATS pss
JOIN PFL_DB.STATS_AND_INFO.PLAYERS p ON pss.PLAYER_ID = p.PLAYER_ID
JOIN PFL_DB.STATS_AND_INFO.TEAMS t ON pss.TEAM_ID = t.TEAM_ID
JOIN PFL_DB.STATS_AND_INFO.PLAYER_CONTRACTS pc ON pss.PLAYER_ID = pc.PLAYER_ID AND pc.IS_ACTIVE = TRUE
GROUP BY p.FIRST_NAME, p.LAST_NAME, t.FULL_NAME, pc.TOTAL_VALUE, pc.ANNUAL_SALARY
ORDER BY TOTAL_PASS_YARDS DESC
LIMIT 5;

Ok... maybe typing that prompt into CoCo did take longer than manually copy and pasting. But this illustrates a concept that we will get into. Snowflake CoCo doesn't just answer questions about your data. It can also edit and create things like Notebooks for you.

## Key Takeaways

In this lab, you explored how Snowflake CoCo can be used to:

- Explore your Data Environment
- Query your data without writing any SQL code
- Use `@` mentions to reference Snowflake objects